# Module 3 — Demo lab 1: R tools for descriptive EDA

Not graded · Standalone walkthroughs for [Project 3](../projects/project-3).

Prerequisites: [Read 1](../lectures/module-3-read-01), [Read 2](../lectures/module-3-read-02).

| Tool | Variables | When |
|------|-----------|------|
| `count()` + proportions | 1 categorical | One categorical variable |
| `group_by()` + `summarize()` | 1 numerical + 1 categorical | Numerical by group |
| `geom_bar`, `geom_histogram`, `geom_boxplot`, `geom_point` | 1–2, any type | Five EDA situations |
| `cor()` | 2 numerical | Linear relationship between two numerical |
| `read_csv()` | whole dataset | Load TidyTuesday / CSV URLs |


## Code anatomy legend (demo notebooks only)

As you read through the demo and see R code, try to break it down into the following 4 components: 
| Color | Meaning in code |
|-------|-----------------|
| <span style="color:#2563eb">Blue</span> | R functions and syntax (`mean`, `<-`, `()`, `~`) |
| <span style="color:#059669">Green</span> | Names you created (variables, data frames) |
| <span style="color:#dc2626">Red</span> | Values to change for your question or data |
| <span style="color:#7c3aed">Purple</span> | Important output to read carefully |


*Note:* Colab may highlight R syntax in its own colors. My anatomy colors appear only on my website, not in Colab.


------------------------------------------------------------------------

## Demo 1 — Tools

Read this demo. Then open [Demo 2](module-3-lab-demo-worked) for five worked examples.

For every chunk, practice spotting the parts you will change for your own analysis: the dataset name and each variable's name and type (check types with `glimpse()`). An "Identify" prompt with the answer follows each code chunk: these are exactly the pieces you will swap in the exercise lab and Project 3.


## A little about the tidyverse

The tidyverse is a family of R packages that share one design philosophy. It grew out of ggplot2, Hadley Wickham's take on the *Grammar of Graphics*, the idea that almost any statistical graphic can be assembled from a few reusable pieces. You map the columns you have to the visual properties you want to see inside `aes()` (`aes(x = ..., y = ..., color = ...)`), then add layers with `+`: a geometry (`geom_point()`, `geom_bar()`), statistical transforms (binning for a histogram, smoothing), scales, and faceting (`facet_wrap()`) to split one plot into small multiples. Each `+` stacks another layer on top, so even a busy figure reads as a short list of simple instructions.

That same "build it from small steps" spirit carries into data wrangling through the pipe. A pipe takes the result on its left and feeds it in as the first argument to the function on its right, so a pipeline reads top-to-bottom in the order the work actually happens: *take the data, then group it, then summarize it, then arrange it.* This mirrors how we describe the task in plain language, and it is close to SQL: `select()`, `filter()`, `group_by()`, and `summarize()` do the jobs of `SELECT`, `WHERE`, and `GROUP BY`, just written as a sequence of steps. You will meet two pipe symbols in the wild: base R's native pipe `|>` (R 4.1+) and the magrittr pipe `%>%` (loaded with the tidyverse). They behave the same for our purposes. This course uses the native `|>` everywhere — including the [R Quick Reference](../r-reference) — so use `|>` in your own code too.

For complete explanations, full function references, and worked examples, see the official tidyverse site: [tidyverse.org](https://www.tidyverse.org/). The reference pages for [dplyr](https://dplyr.tidyverse.org/) (the data verbs like `count()`, `group_by()`, and `summarize()`) and [ggplot2](https://ggplot2.tidyverse.org/) (every geom and aesthetic) are the most useful to keep open while you work.

A few differences from base R are worth noting. Inside tidyverse verbs you refer to columns by their bare names: `summarize(mean_mass = mean(body_mass_g))`,  rather than the base-R `dataframe$column` or `dataframe[["column"]]` form. You rarely need `$`, explicit loops, or repeated `dataframe$` prefixes, because each verb already knows which data frame it is working on. And every verb returns a new data frame (leaving the original unchanged), which is what makes a pipeline easy to read, reorder, and debug one line at a time.

## Setup

```{r}
library(tidyverse)
if (!requireNamespace("palmerpenguins", quietly = TRUE)) {
  install.packages("palmerpenguins", repos = "https://cloud.r-project.org")
}
library(palmerpenguins)
```

Demo 1 goal: learn each tools, that is, the verbs and geoms you need for [Project 3](../projects/project-3). Then open [Demo 2](module-3-lab-demo-worked) for five full worked problems.


## Getting your data into R: CSV files, tibbles, and missing values

Before the tools, here is the groundwork for loading data and getting it ready to analyze. 

#### What a CSV file is

A CSV ("comma-separated values") file is a plain-text spreadsheet: each line is a row, and commas separate the columns. It is the same idea as an Excel sheet, but stripped down to text so any program can read it. `read_csv("...")` (from the tidyverse) reads a CSV (from a file on your computer or a URL) into R. Base R also has `read.csv()`; we use `read_csv()` because it is faster, clearer about column types, and returns a tibble.

#### What "tidy" data looks like

A tidy dataset is organized so that each row is one observation (one penguin, one student, one game) and each column is one variable (a single recorded attribute). Every cell holds one value. Project 3 expects tidy data, and all the tidyverse verbs and plots assume this rows-and-columns shape.

#### Tibble vs. data.frame

`read_csv()` returns a tibble. A tibble is the tidyverse's version of the `data.frame` you met in Module 1: it stores the same rows-and-columns table and behaves the same way in our verbs and plots. The differences are conveniences: a tibble prints only the first 10 rows and shows each column's type at the top, it never silently turns text into factors, and subsetting it always returns another tibble. For everything in this course you can treat a tibble exactly like a data frame.

#### Inspect with glimpse()

`glimpse(dat)` gives a sideways overview of the data: one line per column showing the column name, its type, and its first few values. The type codes are `<dbl>` (numeric), `<int>` (integer), `<chr>` (text), `<fct>` (factor / categorical), and `<lgl>` (TRUE/FALSE). Run `glimpse()` right after loading to confirm the file came in and to check that each variable has the type you expect.

#### Check and convert variable types (Module 1 reminder)

A variable's type decides which analysis is valid, so verify it before you plot or summarize. Check one column with `class(dat$variable)` or all columns with `glimpse(dat)`. If a type is wrong, for example a category stored as text, or a number read in as text, thten convert it and store the result back with `mutate()`:

- `as.factor(x)` or `factor(x, levels = ...)` — text → categorical (use `levels` to set the order)
- `as.numeric(x)` — text → number
- `as.integer(x)` — whole numbers
- `as.character(x)` — anything → text

#### Missing data

Real datasets have gaps. R represents a missing value as `NA`. In the raw CSV those gaps may show up as a blank cell, the text `NA` or `N/A`, an empty string, or a sentinel code like `-99`. `read_csv()` treats blanks and `NA` as missing automatically, but watch for odd codes and fix them. Count missing values with `sum(is.na(dat$variable))`, or check several columns at once with `summarize(across(c(...), ~ sum(is.na(.))))`.

#### Complete-case analysis

For Project 3 we use complete cases: rows with no missing values in the variables we are studying. `drop_na()` (optionally naming the columns) removes any row that still has an `NA`, leaving a clean dataset to summarize and plot.

#### Best practice: never edit the raw file

Do not hand-edit the original CSV. Keep the raw data exactly as downloaded, do every cleaning step in code, and store the result in a new object with a new name — for example `dat_clean <- dat |> ... |> drop_na()`. This keeps your work reproducible (anyone can rerun your code on the original file and get the same clean data) and lets you trace every change. If you need the cleaned data on disk, write a separate copy with `write_csv(dat_clean, "yourdata_clean.csv")` instead of overwriting the original.

In [ ]:
# Check variable types
class(penguins$species)        # one column
glimpse(penguins)              # every column at once

# Convert types if needed, storing the result in a NEW object (leave `penguins` untouched)
penguins_typed <- penguins |>
  mutate(
    species = as.factor(species),   # categorical -> factor; set order with factor(..., levels = ...)
    year    = as.integer(year)      # store year as a whole number
  )

glimpse(penguins_typed)

Identify before you reuse this: the dataset, the new object you store the result in, and each variable you convert with its current and target type. Confirm with `glimpse(penguins)`.

Answer: dataset `penguins`, stored as the new object `penguins_typed` (the original is left unchanged); `species` kept as a factor (categorical); `year` set to integer (whole number).

### Tool 1 — `count()` and proportions

Variables: one categorical.

When: you have one categorical variable and want to know how many rows fall in each category.

Read the pipeline in the next cell top-to-bottom: *take `penguins`, then count by species, then add a proportion column.*

- `count(species)` tallies the rows for each value of `species` and returns a small new data frame: one row per category plus a column called `n` (the count).
- `mutate(prop = n / sum(n))` adds a new column without removing anything. `sum(n)` is the total of all the counts, so `n / sum(n)` is the share in each category. The `prop` values add up to 1.

`|>` is the pipe: it takes the result on its left and feeds it as the first argument to the function on its right, so each line is one step of the work.


In [ ]:
penguins |>
  count(species) |>
  mutate(prop = n / sum(n))


Identify before you reuse this: the dataset name, and the variable with its type. Confirm types with `glimpse(penguins)`.

Answer: dataset `penguins`; variable `species` — categorical.

### Tool 2 — `group_by()` + `summarize()`

Variables: two — one numerical (the variable you summarize) and one categorical (the grouping variable).

When: you want one summary number (or several) for each group.

`summarize()` collapses many rows down to a single row of summaries. On its own it summarizes the whole data frame; placed *after* `group_by(species)`, it runs once per group and returns one row per species.

Inside `summarize()` you name each result yourself (the name on the left of `=` is your choice):

- `n = n()` — `n()` counts the rows in each group.
- `med = median(flipper_length_mm, na.rm = TRUE)` — the median flipper length.
- `iqr = IQR(flipper_length_mm, na.rm = TRUE)` — the interquartile range.

`na.rm = TRUE` tells R to ignore missing values (`NA`) when computing; without it, a single `NA` makes the whole result `NA`. You can use the bare column name `flipper_length_mm` (no `penguins$`) because `summarize()` already knows it is working on the piped-in data.


In [ ]:
penguins |>
  group_by(species) |>
  summarize(
    n = n(),
    med = median(flipper_length_mm, na.rm = TRUE),
    iqr = IQR(flipper_length_mm, na.rm = TRUE)
  )


Identify before you reuse this: the dataset name, and each variable with its type. Confirm types with `glimpse(penguins)`.

Answer: dataset `penguins`; `species` — categorical (the grouping variable); `flipper_length_mm` — numerical (the variable summarized).

### Tool 3 — Core `ggplot2` geoms

Variables: one or two, any type — pick the geom that matches your situation (the table lists the variable count and type for each).

| Situation | Geom |
|-----------|------|
| One categorical | `geom_bar()` |
| One numerical | `geom_histogram()`, `geom_dotplot()` |
| Numerical × categorical | `geom_boxplot()` |
| Two numerical | `geom_point()` |
| Two categorical | `geom_bar(position = "fill")` |

How to read a `ggplot2` call: every plot below has the same shape, `ggplot(data, aes(...)) + geom_*()`.

- `ggplot(penguins, ...)` says which data frame to use.
- `aes(x = species, y = flipper_length_mm, fill = species)` maps columns to visual roles — the x-axis, y-axis, and fill (color).
- `+ geom_*()` adds the layer that actually draws something. The `+` stacks layers, so you read a plot as a short list of instructions.

So `ggplot(penguins, aes(x = species)) + geom_bar()` reads as: *use `penguins`, put `species` on the x-axis, draw bars.* In the last plot, `position = "fill"` rescales the stacked bars so each one reaches 100%, turning counts into within-group proportions.

`geom_bar()` vs `geom_col()`: `geom_bar()` tallies rows for you (`stat = "count"`), so it is easiest with raw data which has one row per observation. When you want category counts (the usual Project 3 case). `geom_col()` draws bars at heights you supply (`stat = "identity"`), so it is easiest when you already have a table of values — e.g. the probability distribution in Project 1, where each outcome already has its `P(X = x)`. Rule of thumb: raw data → `geom_bar()`; a precomputed `x`/`y` table → `geom_col()`.


In [ ]:
ggplot(penguins, aes(x = species)) + geom_bar()
ggplot(penguins, aes(x = flipper_length_mm)) + geom_histogram(binwidth = 5)
ggplot(penguins, aes(x = species, y = flipper_length_mm)) + geom_boxplot()
ggplot(penguins, aes(x = bill_length_mm, y = flipper_length_mm)) + geom_point()
ggplot(penguins, aes(x = island, fill = species)) + geom_bar(position = "fill")


Identify before you reuse this: the dataset name, and every variable mapped inside `aes()` with its type. Confirm types with `glimpse(penguins)`.

Answer: dataset `penguins`; `species` — categorical; `island` — categorical; `flipper_length_mm` — numerical; `bill_length_mm` — numerical.

### Tool 4 — Pearson `cor()` (linear scatterplots only)

Variables: two numerical.

When: two numerical variables that look roughly linear on a scatterplot.

`cor(x, y)` returns Pearson's correlation `r`: a single number between -1 and 1 that measures the strength and direction of a *linear* relationship (near 0 = weak, near +1 or -1 = strong). Only trust it when the scatterplot looks like a line, not a curve.

`filter(!is.na(bill_length_mm), !is.na(flipper_length_mm))` keeps only the rows where both variables are present. `is.na(x)` is `TRUE` when a value is missing, and `!` flips it — so `!is.na(x)` means "x is not missing." We drop those rows first because `cor()` returns `NA` if any value is missing.


In [ ]:
penguins |>
  filter(!is.na(bill_length_mm), !is.na(flipper_length_mm)) |>
  summarize(r = cor(bill_length_mm, flipper_length_mm))


Identify before you reuse this: the dataset name, and the two variables with their types. Confirm types with `glimpse(penguins)`.

Answer: dataset `penguins`; `bill_length_mm` — numerical; `flipper_length_mm` — numerical. (`cor()` needs two numerical variables.)

### Tool 5 — Load data, inspect, and check missingness

Variables: the whole dataset — this is the import and cleaning step before you analyze any single variable.

Project 3 starts here: load your CSV by URL with `read_csv()`, confirm it with `glimpse()`, then count missing values and (if needed) build a clean dataset to summarize and plot.

Walking through the code:

- `glimpse(pen_csv)` prints one line per column — its name, type, and first few values — so you can confirm the file loaded.
- `summarize(across(c(...), ~ sum(is.na(.))))` checks several columns at once. For each column you list, it computes `sum(is.na(.))`, where `.` stands for "this column" and `is.na()` flags the missing entries — so the sum is the number of `NA`s in that column.
- `select(species, island, flipper_length_mm, body_mass_g)` keeps only the columns you name.
- `drop_na()` removes any row that still has a missing value, leaving a clean data frame you can summarize and plot.


In [ ]:
# Load any CSV by URL (TidyTuesday penguins shown — replace with your dataset)
pen_csv <- read_csv(
  "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-07-28/penguins.csv"
)

glimpse(pen_csv)                       # confirm the file loaded

# Count missing values in the variables you plan to use
pen_csv |>
  summarize(across(c(species, island, flipper_length_mm, body_mass_g),
                   ~ sum(is.na(.))))

# Keep your chosen variables and drop rows with any missing value
pen_clean <- pen_csv |>
  select(species, island, flipper_length_mm, body_mass_g) |>
  drop_na()

glimpse(pen_clean)


Identify before you reuse this: the dataset name(s), and each variable with its type. Confirm types with `glimpse(pen_csv)`.

Answer: datasets `pen_csv` (raw) and `pen_clean` (after `select()` + `drop_na()`); `species` — categorical; `island` — categorical; `flipper_length_mm` — numerical; `body_mass_g` — numerical.